<a href="https://www.kaggle.com/code/nihalabhay/chest-imagenet?scriptVersionId=339990628" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# ===== MOBILENET SESSION: FULL REBUILD + RUNS 7-9/12, ALL 3 SOURCES, BOTH PHASES, UNATTENDED =====
!pip install -q tensorflow==2.19.0

import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'

import random
import numpy as np
import pandas as pd
import glob
import tensorflow as tf
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print(f"Seed {SEED} set, TF {tf.__version__}, tf.keras module: {tf.keras.__name__}")

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0, MobileNetV2, ResNet50
from tensorflow.keras.applications.efficientnet import preprocess_input as eff_pre
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mob_pre
from tensorflow.keras.applications.resnet50 import preprocess_input as res_pre
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import (Input, Conv2D, BatchNormalization, MaxPooling2D,
                                      Dropout, GlobalAveragePooling2D, Dense)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, CSVLogger
from tensorflow.keras.metrics import AUC
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, confusion_matrix

CHEX_DIR      = '/kaggle/input/datasets/ashery/chexpert/'
CHEX_CSV      = '/kaggle/input/datasets/ashery/chexpert/train.csv'
NIH_DIR       = '/kaggle/input/datasets/organizations/nih-chest-xrays/data/'
NIH_CSV       = '/kaggle/input/datasets/organizations/nih-chest-xrays/data/Data_Entry_2017.csv'
VINBIG_CSV    = '/kaggle/input/competitions/vinbigdata-chest-xray-abnormalities-detection/train.csv'
VINBIG_PNG    = '/kaggle/input/datasets/xhlulu/vinbigdata-chest-xray-png-512px-original-ratio/train/'
WEIGHTS_DIR   = '/kaggle/input/datasets/ab0y04/all-best-models/'
CHEST_WEIGHTS_DIR = '/kaggle/input/datasets/nihalabhay/best-models/'
WORK_DIR      = '/kaggle/working/'
FINAL_CLASSES = ['no_finding', 'pathology']
NUM_CLASSES   = 2
IMG_SIZE   = 224
BATCH_SIZE = 32
SUBSAMPLE_SEED = 42
CV_SEEDS      = [42, 123, 456, 789, 1024]
TARGET_PER_SOURCE = 15000
PHASE1_EPOCHS = 10
PHASE1_LR     = 1e-3
PHASE2_LR     = 1e-5
CUSTOM_LR     = 1e-3
EARLYSTOP_PAT = 7
MONITOR       = 'val_accuracy'
AUG = dict(rotation_range=20, width_shift_range=0.1, height_shift_range=0.1,
           horizontal_flip=True, zoom_range=0.1)
W_CUSTOM = 'best_custom_cnn_chest.keras'
W_EFF    = 'best_efficientnet_chest.keras'
W_MOB    = 'best_mobilenet_chest.keras'
W_RES    = 'best_resnet50_chest.keras'

chex = pd.read_csv(CHEX_CSV)
chex['label'] = np.where(chex['No Finding'] == 1.0, 'no_finding', 'pathology')
chex['patient_id'] = chex['Path'].str.extract(r'(patient\d+)')
chex['image_path'] = CHEX_DIR + chex['Path'].str.replace('CheXpert-v1.0-small/', '', regex=False)
chex['source'] = 'chex'
assert chex['patient_id'].isna().sum() == 0
_s = chex['image_path'].sample(200, random_state=SEED); assert _s.apply(os.path.exists).sum() == 200

nih = pd.read_csv(NIH_CSV)
nih['label'] = np.where(nih['Finding Labels'] == 'No Finding', 'no_finding', 'pathology')
nih['patient_id'] = nih['Patient ID'].astype(str)
nih['source'] = 'nih'
nih_files = glob.glob(os.path.join(NIH_DIR, '**', '*.png'), recursive=True)
nih_map = {os.path.basename(p): p for p in nih_files}
nih['image_path'] = nih['Image Index'].map(nih_map)
assert nih['image_path'].isna().sum() == 0

vin_raw = pd.read_csv(VINBIG_CSV)
img_findings = vin_raw.groupby('image_id')['class_id'].apply(lambda s: set(s))
vin = pd.DataFrame({'image_id': img_findings.index})
vin['label'] = img_findings.apply(lambda fs: 'no_finding' if fs == {14} else 'pathology').values
vin['image_path'] = VINBIG_PNG + vin['image_id'] + '.png'
vin['source'] = 'vinbig'; vin['patient_id'] = None
_s = vin['image_path'].sample(200, random_state=SEED); assert _s.apply(os.path.exists).sum() == 200

def subsample_by_patient(df, target_n, seed=SUBSAMPLE_SEED):
    pats = df['patient_id'].drop_duplicates().sample(frac=1.0, random_state=seed).tolist()
    sizes = df['patient_id'].value_counts()
    chosen, count = [], 0
    for p in pats:
        n = sizes[p]
        if count + n > target_n and count >= target_n * 0.98: break
        chosen.append(p); count += n
        if count >= target_n: break
    return df[df['patient_id'].isin(chosen)].reset_index(drop=True)

chex_s = subsample_by_patient(chex, TARGET_PER_SOURCE)
nih_s  = subsample_by_patient(nih,  TARGET_PER_SOURCE)
vin_s  = vin.copy()
for full, sub in [(chex, chex_s), (nih, nih_s), (vin, vin_s)]:
    assert abs((full['label']=='pathology').mean() - (sub['label']=='pathology').mean()) < 0.03

cols = ['image_path', 'label', 'source', 'patient_id']
master_df = pd.concat([chex_s[cols], nih_s[cols], vin_s[cols]], ignore_index=True)

def safe_split(df, label_col, test_size, rs, tag=""):
    try:
        return train_test_split(df, test_size=test_size, stratify=df[label_col], random_state=rs)
    except ValueError as e:
        print(f"WARNING [{tag}]: stratified split failed, unstratified fallback. {e}")
        return train_test_split(df, test_size=test_size, random_state=rs)

def split_patient_level(df, rs=SEED, tag=""):
    pg = df.groupby('patient_id')['label'].agg(lambda s: s.value_counts().index[0]).reset_index()
    p_tr, p_tmp = safe_split(pg, 'label', 0.30, rs, tag=f"{tag} first")
    p_va, p_te  = safe_split(p_tmp, 'label', 0.50, rs, tag=f"{tag} second")
    pick = lambda ids: df[df['patient_id'].isin(ids['patient_id'])]
    return (pick(p_tr), pick(p_va), pick(p_te), set(p_tr['patient_id']), set(p_va['patient_id']), set(p_te['patient_id']))

def split_image_level(df, rs=SEED, tag="VinBig"):
    tr, tmp = safe_split(df, 'label', 0.30, rs, tag=f"{tag} first")
    va, te  = safe_split(tmp, 'label', 0.50, rs, tag=f"{tag} second")
    return tr, va, te

c_tr, c_va, c_te, cs_tr, cs_va, cs_te = split_patient_level(chex_s, tag="CheXpert")
n_tr, n_va, n_te, ns_tr, ns_va, ns_te = split_patient_level(nih_s,  tag="NIH")
v_tr, v_va, v_te = split_image_level(vin_s)
assert cs_tr.isdisjoint(cs_te) and cs_tr.isdisjoint(cs_va) and cs_va.isdisjoint(cs_te)
assert ns_tr.isdisjoint(ns_te) and ns_tr.isdisjoint(ns_va) and ns_va.isdisjoint(ns_te)

train_df = pd.concat([c_tr, n_tr, v_tr], ignore_index=True)
val_df   = pd.concat([c_va, n_va, v_va], ignore_index=True)
test_df  = pd.concat([c_te, n_te, v_te], ignore_index=True)

cls = np.array(FINAL_CLASSES)
cw  = compute_class_weight('balanced', classes=cls, y=train_df['label'])
CLASS_WEIGHT = {i: w for i, w in enumerate(cw)}

def make_gens(preprocess_fn, tr_df=None, va_df=None, te_df=None):
    tr_df = train_df if tr_df is None else tr_df
    va_df = val_df   if va_df is None else va_df
    te_df = test_df  if te_df is None else te_df
    if preprocess_fn is None:
        train_idg = ImageDataGenerator(rescale=1./255, **AUG)
        eval_idg  = ImageDataGenerator(rescale=1./255)
    else:
        train_idg = ImageDataGenerator(preprocessing_function=preprocess_fn, **AUG)
        eval_idg  = ImageDataGenerator(preprocessing_function=preprocess_fn)
    common = dict(x_col='image_path', y_col='label', target_size=(IMG_SIZE,IMG_SIZE), batch_size=BATCH_SIZE,
                  class_mode='categorical', classes=FINAL_CLASSES, color_mode='rgb')
    tr = train_idg.flow_from_dataframe(tr_df, shuffle=True,  seed=SEED, **common)
    va = eval_idg.flow_from_dataframe(va_df,  shuffle=False, **common)
    te = eval_idg.flow_from_dataframe(te_df,  shuffle=False, **common)
    return tr, va, te

def build_pretrained(base_class, num_classes=2, shape=(224,224,3)):
    base = base_class(include_top=False, weights='imagenet', input_shape=shape)
    model = Sequential([base, GlobalAveragePooling2D(), Dense(256,activation='relu'), Dropout(0.3), Dense(num_classes,activation='softmax')])
    return model, base

source_splits = {
    'chex':   (c_tr, c_va, c_te),
    'nih':    (n_tr, n_va, n_te),
    'vinbig': (v_tr, v_va, v_te),
}
stage11_results = []   # NOTE: CustomCNN's 3 rows + EfficientNetB0's 3 rows from earlier sessions
                        # are not in this list, re-add all 6 manually before Stage 11's final CSV export

print(f"\nPooled Train {len(train_df):,} | Val {len(val_df):,} | Test {len(test_df):,}")
print("Rebuild complete.\n")

# ===== RUNS 7-9/12: MOBILENETV2, ALL 3 SOURCES, BOTH PHASES, UNATTENDED =====
sources_to_run = ['chex', 'nih', 'vinbig']

for src in sources_to_run:
    print(f"\n{'='*20} MOBILENETV2 / {src.upper()} {'='*20}")
    tr_s, va_s, te_s = source_splits[src]
    tr, va, te = make_gens(mob_pre, tr_df=tr_s, va_df=va_s, te_df=te_s)
    cw_src = compute_class_weight('balanced', classes=np.array(FINAL_CLASSES), y=tr_s['label'])
    cw_dict = {i: w for i, w in enumerate(cw_src)}
    print(f"class_weight ({src}-only):", {c: round(w,3) for c,w in zip(FINAL_CLASSES, cw_src)})

    mob_model, mob_base = build_pretrained(MobileNetV2)
    mob_base.trainable = False
    mob_model.compile(Adam(PHASE1_LR), 'categorical_crossentropy', ['accuracy', AUC(name='auc')])
    p1_path = f'/kaggle/working/ss_mob_{src}_phase1.keras'
    cbs1 = [ModelCheckpoint(p1_path, monitor='val_auc', mode='max', save_best_only=True),
            CSVLogger(f'/kaggle/working/ss_mob_{src}_phase1_log.csv', append=False)]
    mob_model.fit(tr, validation_data=va, epochs=PHASE1_EPOCHS, class_weight=cw_dict, callbacks=cbs1, verbose=1)
    print(f"Phase 1 ({src}) done.")

    mob_model = load_model(p1_path)
    mob_model.layers[0].trainable = True
    mob_model.compile(Adam(PHASE2_LR), 'categorical_crossentropy', ['accuracy', AUC(name='auc')])
    p2_path = f'/kaggle/working/ss_mob_{src}_best.keras'
    cbs2 = [EarlyStopping(monitor='val_auc', mode='max', patience=EARLYSTOP_PAT, restore_best_weights=True),
            ModelCheckpoint(p2_path, monitor='val_auc', mode='max', save_best_only=True),
            CSVLogger(f'/kaggle/working/ss_mob_{src}_phase2_log.csv', append=False)]
    mob_model.fit(tr, validation_data=va, epochs=60, class_weight=cw_dict, callbacks=cbs2, verbose=1)

    preds = mob_model.predict(te, verbose=0)
    proba = preds[:,1]; y = np.array(te.classes); yhat = (proba>=0.5).astype(int)
    auc = roc_auc_score(y, proba); acc = accuracy_score(y, yhat); f1 = f1_score(y, yhat, average='macro')
    cm = confusion_matrix(y, yhat); tn,fp,fn,tp = cm.ravel()
    sens = tp/(tp+fn); spec = tn/(tn+fp) if (tn+fp) > 0 else float('nan')
    print(f"\n>>> MobileNetV2 / {src} TEST: AUC={auc:.4f} Acc={acc:.4f} MacroF1={f1:.4f} Sens={sens:.4f} Spec={spec:.4f}")
    print(">>> Confusion [tn,fp,fn,tp]:", tn, fp, fn, tp)

    stage11_results.append({'arch':'MobileNetV2','source':src,'auc':auc,'accuracy':acc,'macro_f1':f1,
                             'sensitivity':sens,'specificity':spec,'tn':tn,'fp':fp,'fn':fn,'tp':tp,
                             'note':'val_auc monitor throughout, unattended 3-source run'})
    del mob_model
    import gc; gc.collect()

print("\n\nALL 3 MOBILENETV2 SINGLE-SOURCE RUNS COMPLETE")
print(pd.DataFrame([r for r in stage11_results if r['arch']=='MobileNetV2']).to_string(index=False))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 645.0/645.0 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 100.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
ydf-tf 2.20.0 requires tensorflow==2.20.0, but you have tensorflow 2.19.0 which is incompatible.
tf-keras 2.20.0 requires tensorflow<2.21,>=2.20, but you have tensorflow 2.19.0 which is incompatible.
tensorflow-text 2.20.1 requires tensorflow<2.21,>=2.20.0, but you have tensorflow 2.19.0 which is incompatible.


2026-08-03 21:08:55.030813: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1785791335.054446      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1785791335.062138      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1785791335.081076      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1785791335.081094      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1785791335.081097      23 computation_placer.cc:177] computation placer alr

Seed 42 set, TF 2.19.0, tf.keras module: tf_keras.api._v2.keras

Pooled Train 31,456 | Val 7,015 | Test 6,505
Rebuild complete.


==================== MOBILENETV2 / CHEX ====================
Found 10354 validated image filenames belonging to 2 classes.
Found 2422 validated image filenames belonging to 2 classes.
Found 2223 validated image filenames belonging to 2 classes.
class_weight (chex-only): {'no_finding': np.float64(5.007), 'pathology': np.float64(0.555)}


I0000 00:00:1785791564.121002      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1785791564.127168      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


9406464/9406464 [==============================] - 0s 0us/step
Epoch 1/10


I0000 00:00:1785791570.363157      74 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1785791572.959495      76 service.cc:152] XLA service 0x7eb4ed2fcbb0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1785791572.959529      76 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1785791572.959533      76 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1785791573.111200      76 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


324/324 [==============================] - 243s 731ms/step - loss: 0.6631 - accuracy: 0.6638 - auc: 0.7157 - val_loss: 0.4945 - val_accuracy: 0.7898 - val_auc: 0.8585
Epoch 2/10
324/324 [==============================] - 160s 493ms/step - loss: 0.5908 - accuracy: 0.7080 - auc: 0.7651 - val_loss: 0.4564 - val_accuracy: 0.8154 - val_auc: 0.8963
Epoch 3/10
324/324 [==============================] - 156s 482ms/step - loss: 0.5723 - accuracy: 0.7027 - auc: 0.7642 - val_loss: 0.5765 - val_accuracy: 0.7159 - val_auc: 0.7736
Epoch 4/10
324/324 [==============================] - 155s 479ms/step - loss: 0.5560 - accuracy: 0.7109 - auc: 0.7817 - val_loss: 0.4695 - val_accuracy: 0.7948 - val_auc: 0.8675
Epoch 5/10
324/324 [==============================] - 155s 478ms/step - loss: 0.5511 - accuracy: 0.7224 - auc: 0.7927 - val_loss: 0.5792 - val_accuracy: 0.6920 - val_auc: 0.7674
Epoch 6/10
324/324 [==============================] - 157s 486ms/step - loss: 0.5576 - accuracy: 0.7208 - auc: 0.7867 - v